# 05 — Selection readout

In the TIL co-culture arm, cells are being killed. A perturbation's
*representation* in that arm relative to control is therefore a phenotype in
its own right — the classic pooled-screen fitness readout — and it costs
almost nothing to compute from data already loaded.

**Why this belongs in this project and not a separate one:** transcriptional
response and survival are not the same phenotype. A screen that measures only
one is blind to the other, and the disagreements between them are the finding.

**Caveat, stated up front:** this is not a properly timecourse-controlled
dropout screen. Differential representation can also reflect infection
efficiency or proliferation differences unrelated to immune pressure. The
control arm partially handles this. Say so rather than overclaiming.

In [ ]:
import sys, warnings
from pathlib import Path
sys.path.insert(0, str(Path.cwd().parent))
warnings.filterwarnings("ignore", category=FutureWarning)

import numpy as np
import pandas as pd
import scanpy as sc
import matplotlib.pyplot as plt

from src.config import load_config, load_panels, paths, set_seed
from src.plotting import apply_style, condition_palette, savefig

cfg = load_config()
panels = load_panels()
P = paths(cfg)
SEED = set_seed(cfg)
apply_style(cfg)

sc.settings.verbosity = 1
print(f"repo: {P.root}")
print(f"seed: {SEED}")


In [ ]:
import mudata as md
mdata = md.read(P.data_interim / "frangieh_qc.h5mu")
rna = mdata["rna"]
s = cfg["schema"]["obs"]

## 1. Abundance shift per perturbation

In [ ]:
from src.stats import guide_enrichment

enrich = guide_enrichment(rna, cfg)
enrich.to_csv(P.tables / "05_selection_enrichment.csv", index=False)
enrich.head(15)

In [ ]:
alpha = cfg["selection"]["fdr_alpha"]
coculture = [c for c in enrich["condition"].unique() if "ultur" in str(c)]
cond = coculture[0] if coculture else enrich["condition"].iloc[0]
g = enrich[enrich["condition"] == cond]

fig, ax = plt.subplots(figsize=(7, 5.5))
sig_mask = g["padj"] < alpha
ax.scatter(g.loc[~sig_mask, "log2_odds_ratio"], -np.log10(g.loc[~sig_mask, "pvalue"]),
           s=16, c="lightgrey")
ax.scatter(g.loc[sig_mask, "log2_odds_ratio"], -np.log10(g.loc[sig_mask, "pvalue"]),
           s=22, c="crimson")
ax.axvline(0, ls="--", c="k", lw=1)
ax.set_xlabel(f"log2 odds ratio  ({cond} vs reference)\n<- depleted (sensitiser)   enriched (evasion) ->")
ax.set_ylabel("-log10 p")
ax.set_title(f"Selection readout: {cond}")
savefig(fig, "05_selection_volcano", cfg)

## 2. Join the two readouts

Transcriptional effect size (E-distance in co-culture) against selection
log-odds. The quadrants:

- **depleted + strong response** — sensitiser: responding, and dying anyway
- **enriched + strong response** — candidate evasion mechanism
- **enriched + no response** — evasion *without* a transcriptional signature.
  The most surprising quadrant, and the one worth dwelling on.
- **neither** — likely a non-functional guide

In [ ]:
# TODO: merge enrich with the E-distance table from nb03 on
# [perturbation, condition]; scatter, label quadrants, annotate outliers
# with adjustText.
print("[stub] two-readout comparison")

## 3. Wrap up

Three claims constitute "done":

1. A ranked, defensible list of context-dependent perturbations, with
   IFN-γ/JAK-STAT and antigen presentation recovered as positive controls.
2. An RNA-vs-protein quadrant analysis that independently rediscovers the CD58
   discordance.
3. A demonstration that transcriptional and selection readouts disagree, with a
   worked example.

If those exist and the limitations section is honest, stop. **Resist adding a
prediction model** — that is the next project.

In [ ]:
import session_info
session_info.show()